# Lab 9 - Meeting Notes Action Plan with an Outcome Rubric

**Estimated time:** 20min | **Estimated cost:** $0.06

![Architecture](assets/architecture.svg)

## Goal
Build an outcome-driven session that turns messy meeting notes into a structured Markdown action plan. Instead of chatting and deciding for yourself when the plan is good enough, you hand the agent a goal and a rubric and let a separate grader score its work until it passes.

A conversational session is "you talk, it responds, you decide when to stop." An outcome-driven session is "you define done, it iterates, and a separate grader decides when it is done." You send a single `user.define_outcome` event carrying a **description** of the goal and a **gradeable rubric**. A managed grader scores each iteration and feeds per-criterion gaps back to the builder. The session loops build -> grade -> revise until the grader returns `satisfied`, hits `max_iterations`, or is interrupted.

This lab is intentionally small and inexpensive. It creates a lightweight cloud environment, mounts `meeting_notes.md`, and asks the agent to write `/mnt/session/outputs/action_plan.md`.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`.
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.
- The sample `meeting_notes.md` file in this lab folder.


In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)


In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ").strip()

print("ANTHROPIC_API_KEY configured for this notebook session.")


In [ ]:
import os
from pathlib import Path
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)


### Step 1 - Create the environment and agent
Create a lightweight cloud environment and a meeting action planner agent. No package installs, internet egress, or prebuilt office skills are needed.

In [ ]:
env = client.beta.environments.create(
    name="meeting-notes-outcome",
    config={
        "type": "cloud",
        "networking": {"type": "limited", "allowed_hosts": []},
    },
    betas=BETAS,
)
print("env.id =", env.id)

agent = client.beta.agents.create(
    name="Meeting Action Planner",
    model=MODEL,
    system=(
        "You are an operations lead. Turn messy meeting notes into concise, "
        "source-grounded action plans. Do not invent owners, dates, or "
        "decisions. Write deliverables to /mnt/session/outputs/."
    ),
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
print("agent.id =", agent.id)


### Step 2 - Upload and mount `meeting_notes.md`
Upload the source notes and mount them at a stable path inside the session.

In [ ]:
notes = client.beta.files.upload(file=Path("meeting_notes.md"), betas=BETAS)
print("file.id =", notes.id)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=env.id,
    resources=[{
        "type": "file",
        "file_id": notes.id,
        "mount_path": "/workspace/meeting_notes.md",
    }],
    title="Meeting notes action plan",
    betas=BETAS,
)
print("session.id =", session.id)


### Step 3 - Write a gradeable rubric
The rubric is a Markdown document the grader reads literally. Headings group criteria into categories the grader scores independently; every bullet is one pass/fail rule. Make each bullet testable.

In [ ]:
RUBRIC = """\
# Action Plan Rubric

## Decisions
- The output contains a "Decisions" section with at least 3 decisions from the notes
- The output does not invent decisions that are not supported by the notes

## Action Items
- The output contains an "Action Items" Markdown table with columns Owner, Task, Due Date, Priority, and Source Note
- The table contains at least 6 action items from the notes
- Each action item has a named owner when the notes provide one
- Missing due dates are written as "Needs date" instead of invented dates

## Risks and Open Questions
- The output contains a "Risks" section with at least 2 risks or blockers from the notes
- The output contains an "Open Questions" section with at least 2 unresolved questions from the notes

## Output Quality
- A single Markdown file is written to /mnt/session/outputs/action_plan.md
- The plan is concise, scannable, and uses only information from /workspace/meeting_notes.md
"""

print(RUBRIC)


### Step 4 - Send the outcome and stream grading
There is no `outcome` field on `sessions.create()`: an outcome is an **event** you send next. Open the stream before sending so you do not miss the first grader events. This lab uses `max_iterations=3` to keep cost predictable.

In [ ]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[{
            "type": "user.define_outcome",
            "description": (
                "Read /workspace/meeting_notes.md and create a concise action "
                "plan as Markdown. Include decisions, action items, risks, and "
                "open questions. Write the final file to "
                "/mnt/session/outputs/action_plan.md."
            ),
            "rubric": {"type": "text", "content": RUBRIC},
            "max_iterations": 3,
        }],
        betas=BETAS,
    )

    for event in stream:
        if event.type == "span.outcome_evaluation_start":
            print(f"\n--- grading iteration {event.iteration} ---")
        elif event.type == "span.outcome_evaluation_end":
            print(f"  result: {event.result}")
            if event.result == "needs_revision":
                print(f"  feedback: {event.explanation[:240]}...")
        elif event.type == "agent.tool_use":
            print(f"  [tool: {event.name}]")
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break


### Step 5 - Confirm the final result and pull the deliverable
Read the final outcome result, then list the files written to `/mnt/session/outputs/` and download `action_plan.md` exactly. The session file list can also include the mounted source file `meeting_notes.md`, which is not always downloadable as an output artifact.

In [ ]:
session = client.beta.sessions.retrieve(session.id, betas=BETAS)
for ev in session.outcome_evaluations:
    print(f"outcome {ev.outcome_id}: {ev.result}")

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
downloaded = False
for f in client.beta.files.list(scope_id=session.id, betas=BETAS):
    print(f.id, f.filename)
    if f.filename == "action_plan.md":
        client.beta.files.download(f.id).write_to_file(str(out_dir / f.filename))
        print(f"saved: {out_dir / f.filename}")
        downloaded = True

if not downloaded:
    raise RuntimeError(
        "Could not find downloadable action_plan.md in session outputs. "
        "Confirm the agent wrote /mnt/session/outputs/action_plan.md."
    )


### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
- The first build runs at **iteration 0**.
- Tool calls stream by, usually `read` and `write`.
- The grader emits `span.outcome_evaluation_end` per iteration. You may see an early `needs_revision` if a section or table column is missing.
- The final result is usually `satisfied`, and the session moves to `idle`.
- `action_plan.md` downloads into your local `outputs/` folder.

## Troubleshooting
- **Writing a gradeable rubric.** Every bullet must be a single, testable pass/fail rule. Replace adjectives with measurable specifics.
- **`max_iterations` behavior.** It bounds the build-grade-revise loop. This lab uses 3 to keep cost low.
- **The result taxonomy.** `span.outcome_evaluation_end.result` is one of: `satisfied`, `needs_revision`, `max_iterations_reached`, `failed`, or `interrupted`.
- **`result: failed` immediately.** The rubric and description disagree. Reconcile them and re-send `user.define_outcome` once the session is idle.
- **No `action_plan.md` downloads.** Only files written to `/mnt/session/outputs/` are collected. The session file list can also include the mounted source file `meeting_notes.md`, which is not always downloadable as an output artifact. Filter for `action_plan.md` exactly.